## HERRAMIENTAS OpenWehater y Open Meteo

In [ ]:
"""
demo_openweather_full.py
========================
Demo OpenWeather One Call API 3.0 — timemachine
Variables completas para análisis de accidentalidad vial.

Uso:
    pip install requests pytz
    python demo_openweather_full.py
"""
import requests, pytz, time
from datetime import datetime

BASE_URL        = "https://api.openweathermap.org/data/3.0/onecall/timemachine"
TIMEZONE_BRASIL = pytz.timezone("America/Sao_Paulo")

CASOS = [
    {"id":"742921","fecha":"2026-01-01","hora":"04:04:00","uf":"TO","lat":-7.291548, "lon":-48.286252,"prf":"Céu Claro"},
    {"id":"745723","fecha":"2026-01-15","hora":"07:30:00","uf":"ES","lat":-19.380859,"lon":-40.066511,"prf":"Céu Claro"},
    {"id":"752205","fecha":"2026-02-14","hora":"01:35:00","uf":"RS","lat":-29.882940,"lon":-50.390110,"prf":"Nublado"},
    {"id":"745998","fecha":"2026-01-16","hora":"14:20:00","uf":"MG","lat":-18.893824,"lon":-41.947323,"prf":"Céu Claro"},
    {"id":"752076","fecha":"2026-02-13","hora":"17:00:00","uf":"MG","lat":-21.717847,"lon":-42.730227,"prf":"Céu Claro"},
]

def to_unix(fecha, hora):
    dt = datetime.strptime(f"{fecha} {hora}", "%Y-%m-%d %H:%M:%S")
    return int(TIMEZONE_BRASIL.localize(dt).timestamp())

def get_weather(lat, lon, unix_ts, api_key):
    resp = requests.get(BASE_URL, params={
        "lat": lat, "lon": lon, "dt": unix_ts,
        "appid": api_key, "units": "metric"
    }, timeout=15)
    if resp.status_code != 200:
        return None, f"HTTP {resp.status_code}"
    d = resp.json().get("data", [{}])[0]
    rain = d.get("rain", {})
    wind = d
    return {
        # Precipitación
        "precip_mm"       : rain.get("1h", 0.0),
        "nieve_mm"        : d.get("snow", {}).get("1h", 0.0),
        # Visibilidad
        "visibility_m"    : d.get("visibility", None),
        # Temperatura
        "temp_c"          : d.get("temp", None),
        "sensacion_c"     : d.get("feels_like", None),
        "dew_point_c"     : d.get("dew_point", None),
        # Humedad y presión
        "humedad_pct"     : d.get("humidity", None),
        "presion_hpa"     : d.get("pressure", None),
        # Nubosidad
        "nubes_pct"       : d.get("clouds", None),
        # Viento
        "viento_ms"       : d.get("wind_speed", None),
        "rafaga_ms"       : d.get("wind_gust", None),
        "viento_dir"      : d.get("wind_deg", None),
        # Radiación
        "uvi"             : d.get("uvi", None),
        # Fase del día
        "sunrise_unix"    : d.get("sunrise", None),
        "sunset_unix"     : d.get("sunset", None),
        # Condición
        "cond_main"       : d.get("weather", [{}])[0].get("main", None),
        "cond_desc"       : d.get("weather", [{}])[0].get("description", None),
    }, "OK"

def es_de_dia(unix_ts, sunrise, sunset):
    if sunrise and sunset:
        return "Día" if sunrise <= unix_ts <= sunset else "Noche"
    return "?"

def main():
    print("="*66)
    print("  DEMO — OpenWeather One Call 3.0 — Variables completas")
    print("="*66)
    api_key = input("\n  API key OpenWeather: ").strip()
    if not api_key:
        print("  Sin API key. Saliendo.")
        return

    exitosos = 0
    for caso in CASOS:
        unix_ts = to_unix(caso["fecha"], caso["hora"])
        datos, status = get_weather(caso["lat"], caso["lon"], unix_ts, api_key)

        print(f"\n{'─'*66}")
        print(f"  ID {caso['id']}  |  {caso['fecha']} {caso['hora'][:5]}  |  UF: {caso['uf']}")
        print(f"  PRF: {caso['prf']}  |  Estado: {'✅ OK' if datos else '❌ '+status}")
        if datos:
            exitosos += 1
            fase = es_de_dia(unix_ts, datos['sunrise_unix'], datos['sunset_unix'])
            print(f"\n  {'Variable':<28} {'Valor':<15} Relevancia vial")
            print(f"  {'─'*28} {'─'*15} {'─'*20}")
            rows = [
                ("Precipitación (mm/h)",   datos['precip_mm'],    "★★★ Principal"),
                ("Visibilidad (m)",         datos['visibility_m'], "★★★ Principal"),
                ("Temperatura (°C)",        datos['temp_c'],       "★★  Secundaria"),
                ("Humedad relativa (%)",    datos['humedad_pct'],  "★★  Secundaria"),
                ("Punto de rocío (°C)",     datos['dew_point_c'],  "★★  Secundaria"),
                ("Sensación térmica (°C)",  datos['sensacion_c'],  "★   Contexto"),
                ("Presión (hPa)",           datos['presion_hpa'],  "★   Contexto"),
                ("Cobertura nubosa (%)",    datos['nubes_pct'],    "★   Contexto"),
                ("Viento (m/s)",            datos['viento_ms'],    "★   Contexto"),
                ("Ráfagas (m/s)",           datos['rafaga_ms'],    "★   Contexto"),
                ("Dirección viento (°)",    datos['viento_dir'],   "    Info"),
                ("Índice UV",               datos['uvi'],          "    Info"),
                ("Fase del día",            fase,                  "★★  Secundaria"),
                ("Condición principal",     datos['cond_main'],    "★★★ Principal"),
                ("Descripción",            datos['cond_desc'],    "    Info"),
            ]
            for nombre, valor, rel in rows:
                v = str(valor) if valor is not None else "None ⚠"
                print(f"  {nombre:<28} {v:<15} {rel}")
        time.sleep(0.3)

    print(f"\n{'─'*66}")
    print(f"  RESULTADO: {exitosos}/5 exitosas")
    print("="*66)

if __name__ == "__main__":
    main()


  DEMO — OpenWeather One Call 3.0 — Variables completas

  API key OpenWeather: <OPENWEATHER_API_KEY>

──────────────────────────────────────────────────────────────────
  ID 742921  |  2026-01-01 04:04  |  UF: TO
  PRF: Céu Claro  |  Estado: ✅ OK

  Variable                     Valor           Relevancia vial
  ──────────────────────────── ─────────────── ────────────────────
  Precipitación (mm/h)         0.0             ★★★ Principal
  Visibilidad (m)              None ⚠          ★★★ Principal
  Temperatura (°C)             23.78           ★★  Secundaria
  Humedad relativa (%)         93              ★★  Secundaria
  Punto de rocío (°C)          22.58           ★★  Secundaria
  Sensación térmica (°C)       24.64           ★   Contexto
  Presión (hPa)                1011            ★   Contexto
  Cobertura nubosa (%)         100             ★   Contexto
  Viento (m/s)                 1.8             ★   Contexto
  Ráfagas (m/s)                None ⚠          ★   Contexto
  Dirección 

In [ ]:
"""
demo_openmeteo_full.py
======================
Demo Open-Meteo ERA5 — Variables completas para accidentalidad.
Sin API key. Sin costo.

Uso:
    pip install requests
    python demo_openmeteo_full.py
"""
import requests, time

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

CASOS = [
    {"id":"742921","fecha":"2026-01-01","hora":"04","uf":"TO","lat":-7.291548, "lon":-48.286252,"prf":"Céu Claro"},
    {"id":"745723","fecha":"2026-01-15","hora":"07","uf":"ES","lat":-19.380859,"lon":-40.066511,"prf":"Céu Claro"},
    {"id":"752205","fecha":"2026-02-14","hora":"01","uf":"RS","lat":-29.882940,"lon":-50.390110,"prf":"Nublado"},
    {"id":"745998","fecha":"2026-01-16","hora":"14","uf":"MG","lat":-18.893824,"lon":-41.947323,"prf":"Céu Claro"},
    {"id":"752076","fecha":"2026-02-13","hora":"17","uf":"MG","lat":-21.717847,"lon":-42.730227,"prf":"Céu Claro"},
]

# Variables a solicitar — equivalentes exactas a OpenWeather + extras ERA5
OM_VARS = ",".join([
    "precipitation",           # = rain.1h
    "rain",                    # lluvia sin nieve
    "snowfall",                # nieve
    "visibility",              # = visibility
    "temperature_2m",          # = temp
    "apparent_temperature",    # = feels_like
    "dew_point_2m",            # = dew_point
    "relative_humidity_2m",    # = humidity
    "surface_pressure",        # = pressure
    "cloud_cover",             # = clouds total
    "cloud_cover_low",         # extra — nubes bajas (neblina)
    "wind_speed_10m",          # = wind_speed
    "wind_gusts_10m",          # = wind_gust
    "wind_direction_10m",      # = wind_deg
    "weather_code",            # = weather.main (código WMO)
    "is_day",                  # = sunrise/sunset calculado
    "shortwave_radiation",     # extra — radiación solar W/m²
    "sunshine_duration",       # extra — segundos de sol en la hora
])

# Tabla de códigos WMO → descripción legible
WMO = {
    0:"Cielo despejado", 1:"Principalmente despejado", 2:"Parcialmente nublado",
    3:"Nublado", 45:"Niebla", 48:"Niebla con escarcha",
    51:"Llovizna leve", 53:"Llovizna moderada", 55:"Llovizna intensa",
    61:"Lluvia leve", 63:"Lluvia moderada", 65:"Lluvia intensa",
    71:"Nevada leve", 73:"Nevada moderada", 75:"Nevada intensa",
    77:"Granizo", 80:"Chubascos leves", 81:"Chubascos moderados",
    82:"Chubascos violentos", 85:"Chubascos de nieve leves",
    95:"Tormenta", 96:"Tormenta con granizo leve",
    99:"Tormenta con granizo intenso",
}

def fetch_caso(lat, lon, fecha, hora):
    resp = requests.get(BASE_URL, params={
        "latitude": lat, "longitude": lon,
        "start_date": fecha, "end_date": fecha,
        "hourly": OM_VARS,
        "timezone": "America/Sao_Paulo",
        "wind_speed_unit": "ms",   # m/s para comparar con OW
    }, timeout=20)
    resp.raise_for_status()
    h = resp.json()["hourly"]
    idx = int(hora)
    return {
        # Precipitación
        "precip_mm"      : h["precipitation"][idx],
        "lluvia_mm"      : h["rain"][idx],
        "nieve_mm"       : h["snowfall"][idx],
        # Visibilidad
        "visibility_m"   : h["visibility"][idx],
        # Temperatura
        "temp_c"         : h["temperature_2m"][idx],
        "sensacion_c"    : h["apparent_temperature"][idx],
        "dew_point_c"    : h["dew_point_2m"][idx],
        # Humedad y presión
        "humedad_pct"    : h["relative_humidity_2m"][idx],
        "presion_hpa"    : h["surface_pressure"][idx],
        # Nubosidad
        "nubes_pct"      : h["cloud_cover"][idx],
        "nubes_bajas_pct": h["cloud_cover_low"][idx],
        # Viento
        "viento_ms"      : h["wind_speed_10m"][idx],
        "rafaga_ms"      : h["wind_gusts_10m"][idx],
        "viento_dir"     : h["wind_direction_10m"][idx],
        # Radiación
        "radiacion_wm2"  : h["shortwave_radiation"][idx],
        "sol_segundos"   : h["sunshine_duration"][idx],
        # Fase del día
        "es_dia"         : h["is_day"][idx],
        # Condición
        "weather_code"   : h["weather_code"][idx],
        "cond_desc"      : WMO.get(h["weather_code"][idx], f"WMO {h['weather_code'][idx]}"),
    }

def main():
    print("="*66)
    print("  DEMO — Open-Meteo ERA5 — Variables completas (sin API key)")
    print("="*66)

    exitosos = 0
    for caso in CASOS:
        print(f"\n{'─'*66}")
        print(f"  ID {caso['id']}  |  {caso['fecha']} {caso['hora']}:00  |  UF: {caso['uf']}")
        print(f"  PRF: {caso['prf']}")
        try:
            datos = fetch_caso(caso["lat"], caso["lon"], caso["fecha"], caso["hora"])
            exitosos += 1
            fase = "Día" if datos["es_dia"] == 1 else "Noche"
            print(f"  Estado: ✅ OK\n")
            print(f"  {'Variable':<28} {'Valor':<15} Relevancia vial")
            print(f"  {'─'*28} {'─'*15} {'─'*20}")
            rows = [
                ("Precipitación (mm/h)",   datos['precip_mm'],       "★★★ Principal"),
                ("Visibilidad (m)",         datos['visibility_m'],    "★★★ Principal"),
                ("Temperatura (°C)",        datos['temp_c'],          "★★  Secundaria"),
                ("Humedad relativa (%)",    datos['humedad_pct'],     "★★  Secundaria"),
                ("Punto de rocío (°C)",     datos['dew_point_c'],     "★★  Secundaria"),
                ("Sensación térmica (°C)",  datos['sensacion_c'],     "★   Contexto"),
                ("Presión (hPa)",           datos['presion_hpa'],     "★   Contexto"),
                ("Cobertura nubosa (%)",    datos['nubes_pct'],       "★   Contexto"),
                ("Nubes bajas (%)",         datos['nubes_bajas_pct'], "★   Contexto (niebla)"),
                ("Viento (m/s)",            datos['viento_ms'],       "★   Contexto"),
                ("Ráfagas (m/s)",           datos['rafaga_ms'],       "★   Contexto"),
                ("Dirección viento (°)",    datos['viento_dir'],      "    Info"),
                ("Radiación solar (W/m²)",  datos['radiacion_wm2'],   "★   Contexto"),
                ("Sol en la hora (seg)",    datos['sol_segundos'],    "    Info"),
                ("Fase del día",            fase,                     "★★  Secundaria"),
                ("Condición (WMO)",         datos['cond_desc'],       "★★★ Principal"),
            ]
            for nombre, valor, rel in rows:
                v = str(valor) if valor is not None else "None ⚠"
                print(f"  {nombre:<28} {v:<15} {rel}")
        except Exception as e:
            print(f"  Estado: ❌ Error — {e}")
        time.sleep(0.5)

    print(f"\n{'─'*66}")
    print(f"  RESULTADO: {exitosos}/5 exitosas  |  Costo: $0.00")
    print("="*66)

if __name__ == "__main__":
    main()


  DEMO — Open-Meteo ERA5 — Variables completas (sin API key)

──────────────────────────────────────────────────────────────────
  ID 742921  |  2026-01-01 04:00  |  UF: TO
  PRF: Céu Claro
  Estado: ✅ OK

  Variable                     Valor           Relevancia vial
  ──────────────────────────── ─────────────── ────────────────────
  Precipitación (mm/h)         0.0             ★★★ Principal
  Visibilidad (m)              None ⚠          ★★★ Principal
  Temperatura (°C)             22.0            ★★  Secundaria
  Humedad relativa (%)         95              ★★  Secundaria
  Punto de rocío (°C)          21.1            ★★  Secundaria
  Sensación térmica (°C)       26.4            ★   Contexto
  Presión (hPa)                985.8           ★   Contexto
  Cobertura nubosa (%)         6               ★   Contexto
  Nubes bajas (%)              0               ★   Contexto (niebla)
  Viento (m/s)                 0.05            ★   Contexto
  Ráfagas (m/s)                1.0            